In [2]:
import numpy as np
from matplotlib import pyplot as plt
import csv

%matplotlib inline

import xarray as xr
import lib_ecofun as lef
from importlib import reload
reload(lef)
from scipy.optimize import curve_fit, minimize, dual_annealing

In [3]:
lef.beta_fun(-1, 1)

0.5

# Model tuning

### Setting inicond and obs to fit

In [4]:
fcu = 0.8
inicond = lef.inicond_yr(2000)
inicond['Kf_ini'] = inicond['Kf_ini']*lef.fossil_capacity_util/fcu

obs = dict()
#obs['Ig_ratio'] = lef.Ig_obs/(lef.Ig_obs+lef.If_obs) # considering only investment in power generation capacity (no grids, storage, EVs, ...)
obs['Eg_ratio'] = lef.Eg_ratio

E_obs = xr.load_dataarray('Etot_hist_1965-2022.nc')
E_obs /= E_obs.sel(year = 2000)
obs['E'] = E_obs

### Chose params to fit and bounds (initial guess only needed for minimize)

In [5]:
parnames = ['growth', 'beta_0', 'r_inv', 'a', 'b', 'eta_g', 'eta_f']
bounds = [(0.01, 0.03), (-0.5, 0.5), (0.1, 0.6), (0.5, 1.5), (0.5, 1.5), (0.1, 0.9), (0.1, 0.9)]
# initial_guess = [0.02, 0., 0.1, 1., 1., 0.7, 0.7]

### Set other options: use public inv? what share? what scenario?

In [6]:
year_ini = 2000

params = lef.default_params.copy()
print(params)
print('-------------')
#params['growth'] = 0.029 # fixing Growth!

verbose = True

public_investment = False

params['r_inv_state'] = 0.015
arr = np.concatenate([np.linspace(0., 0.7, 20), np.linspace(0.7, 0.7, 80)])
mu_scen = xr.DataArray(arr, dims = ('year'), coords = {'year': np.arange(year_ini, 2100)})

obs_weights = None

{'growth': 0.01, 'eps': 1, 'a': 1, 'b': 1, 'gamma_f': 0.5, 'gamma_g': 0.5, 'eta_g': 0.2, 'eta_f': 0.2, 'h_g': 0.5, 'h_f': 0.5, 'r_inv': 0.1, 'beta_0': 0.2, 'delta_sig': 0.5, 'delta_g': 0.01, 'delta_f': 0.01, 'f_heavy': 0.1, 'r_inv_state': 0.01}
-------------


In [ ]:
# def cost_function(parset, parnames = ['beta_0', 'gamma_g', 'growth', 'delta_sig'], params = default_params.copy(), year_ini = 2015, inicond = inicond_2015, verbose = False, obs = None, public_investment = False, mu_state_scenario = None, linear_gdp = None, obs_weights = None, break_on_scarcity = False)

result = dual_annealing(lef.cost_function, bounds = bounds, args = (parnames, params, year_ini, inicond, verbose, obs_fit, public_investment, mu_scen, obs_weights))